# DAY 4: Domain-Specific Fine-Tuning (Structured JSON Extraction)

## 🎯 Learning Objectives
This is the **core portfolio fine-tuning pipeline**.
By the end of this day, you will be able to execute and explain:
1. **Domain Data Engineering**: Creating a clean dataset mapping messy, unstructured natural language order text into validated JSON schemas.
2. **Strict Split Strategy & Anti-Leakage**: Partitioning data into train (1,200), validation (150), and held-out test (150) splits with programmatic verification of zero overlap.
3. **End-to-End QLoRA + SFT Pipeline**: Combining 4-bit NF4 base quantization with full-linear LoRA adapter training on a Google Colab T4 GPU.
4. **Training Dynamics & Convergence**: Tracking training and validation loss curves across steps.
5. **Artifact Checkpointing**: Saving the fine-tuned adapter weights and training metadata for evaluation and production serving.

### Step 1: Runtime Environment & Hardware Detection

In [ ]:
# !pip install -q transformers peft trl datasets accelerate bitsandbytes pyyaml

In [ ]:
import os
import sys
import json
import yaml
import torch
import transformers
import peft
import trl

print("=" * 60)
print("🚀 DAY 4: DOMAIN FINE-TUNING PIPELINE")
print("=" * 60)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name        : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print(f"Transformers    : {transformers.__version__}")
print(f"PEFT            : {peft.__version__}")
print(f"TRL             : {trl.__version__}")
print("=" * 60)

### Step 2: Generate Domain-Specific Dataset (with Anti-Leakage Guarantee)

In [ ]:
from src.data.dataset_generator import generate_order_dataset, save_dataset_splits

# Generate 1,200 Train, 150 Validation, and 150 Test samples deterministically (seed=42)
train_set, val_set, test_set = generate_order_dataset(
    total_train=1200,
    total_val=150,
    total_test=150,
    seed=42
)

save_dataset_splits(train_set, val_set, test_set, output_dir="data/processed")
print(f"Sample Training Record:\n{json.dumps(train_set[0], indent=2)}")

### Step 3: Load Base Model in 4-bit NF4 Quantization

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, TaskType

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=compute_dtype
    )
    base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
else:
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

# Attach LoRA adapter
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

### Step 4: Configure Training Arguments & Execute SFT

In [ ]:
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = "models/adapters/qwen-1.5b-order-extractor"
os.makedirs(OUTPUT_DIR, exist_ok=True)

dataset = load_dataset("json", data_files={
    "train": "data/processed/train.jsonl",
    "val": "data/processed/val.jsonl"
})

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,      # Effective batch size = 2 * 4 = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    report_to="none",
    seed=42
)

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    tokenizer=tokenizer,
    max_seq_length=512
)

print("Starting Supervised Fine-Tuning execution...")
train_result = trainer.train()

# Save final adapter
peft_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Training finished! Adapter saved to {OUTPUT_DIR}")

### Step 5: Save Training Metrics to File

In [ ]:
metrics_file = "reports/training_metrics.json"
os.makedirs("reports", exist_ok=True)

metrics_data = {
    "model_id": MODEL_ID,
    "train_loss": train_result.training_loss,
    "train_runtime_seconds": train_result.metrics.get("train_runtime", 0.0),
    "train_samples_per_second": train_result.metrics.get("train_samples_per_second", 0.0),
    "epochs": 3,
    "effective_batch_size": 8,
    "learning_rate": 2e-4
}

with open(metrics_file, "w", encoding="utf-8") as f:
    json.dump(metrics_data, f, indent=2)
print(f"Metrics saved to {metrics_file}:\n{json.dumps(metrics_data, indent=2)}")

---
## 🎓 Day 4 Interview Preparation & Concept Mastery

### 🧠 Concepts You Must Know
1. **Data Leakage in LLM Fine-Tuning**: Occurs if prompt templates, entities, or paraphrased test samples are present in the training set, giving a false illusion of generalization.
2. **Effective Batch Size**: $\text{Per-device batch size} \times \text{Gradient Accumulation Steps} \times \text{Number of GPUs}$.
3. **Convergence Diagnostics**: Observing steady decrease in both train loss and validation loss. If validation loss begins increasing while train loss continues dropping, overfitting has occurred.
4. **Checkpoint Management**: `save_total_limit = 2` ensures only the top 2 best/most recent checkpoints are kept, preventing disk overflow in Colab sessions.

### 💼 5 Interview Questions
1. **Q1 (Beginner)**: *How did you choose your learning rate ($2 \times 10^{-4}$) for LoRA?*
   - **Answer**: Standard full fine-tuning typically uses small learning rates ($10^{-5}$ to $5 \times 10^{-5}$) because modifying all parameters can destabilize pretrained representations. In LoRA, since base weights are frozen and adapter matrices are initialized from scratch, higher learning rates ($10^{-4}$ to $3 \times 10^{-4}$) are necessary for fast adapter convergence.
2. **Q2 (Intermediate)**: *How did you guarantee that your evaluation results would not suffer from data leakage?*
   - **Answer**: We partitioned our dataset generation using deterministic seeds and executed programmatic set-intersection checks on prompt text across `train`, `val`, and `test` splits, confirming zero shared input strings before training.
3. **Q3 (Intermediate)**: *Why did we set `weight_decay = 0.01`?*
   - **Answer**: Weight decay applies L2 regularization to adapter parameter matrices $A$ and $B$, preventing large numerical weights and reducing the risk of overfitting.
4. **Q4 (Advanced)**: *What would happen if we trained for 10 epochs instead of 3 on 1,200 examples?*
   - **Answer**: On a structured extraction dataset of 1,200 samples, 10 epochs (12,000 example exposures) would likely lead to severe memorization and loss of generalization to unseen sentence structures.
5. **Q5 (Advanced)**: *How do you monitor training stability in real-time?*
   - **Answer**: By tracking step-wise loss, gradient norm ($\|\nabla\| \le 1.0$), and periodic validation loss evaluations every 50 steps.

### 🛠️ 3 Practical Questions on Your Code
1. *In your code, where are the adapter weights saved?*
2. *What is the effective batch size computed in your TrainingArguments?*
3. *How did you pass evaluation datasets into SFTTrainer?*

### 🏆 Day 4 Hands-On Challenge
> **Challenge**: Add an evaluation callback or inspect `trainer.state.log_history` to plot the training vs. validation loss curve across steps using `matplotlib`.